# Strategy Comparison: Diversify vs Cluster

Notebook ini membandingkan performa **AI + Graph Diversify** vs **AI + Graph Cluster** dalam strategi V62.

**Tujuan:**
- Membandingkan return, risk, dan risk-adjusted return
- Visualisasi performa kedua strategi per tahun (2023, 2024, 2025)
- Analisis drawdown dan recovery
- Identifikasi strategi mana yang lebih unggul dalam kondisi berbeda

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from scipy.optimize import minimize
import networkx as nx
from sklearn.metrics import fbeta_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('ggplot')
sns.set_palette("husl")

SEED = 42
np.random.seed(SEED)

print("✓ Libraries loaded!")

✓ Libraries loaded!


## 1. Load Data & Setup

In [2]:
# Load cryptocurrency data
file_path = '../experiment_nextLevel2/dataset_2023_2025.xlsx'
data = pd.read_excel(file_path, index_col=0, parse_dates=True)
returns = data.pct_change().dropna()
market_return = returns.mean(axis=1)
market_index = (1 + market_return).cumprod() * 100

print(f"Data: {len(data)} rows, {len(data.columns)} assets")
print(f"Period: {data.index[0]} to {data.index[-1]}")

Data: 990 rows, 25 assets
Period: 2023-04-17 00:00:00 to 2025-12-31 00:00:00


## 2. Train AI Model (Same as V62)

In [3]:
# Prepare features
features = pd.DataFrame(index=returns.index)
features['Vol_20'] = market_return.rolling(window=20).std()
features['Mom_20'] = market_return.rolling(window=20).mean()
features['Mom_50'] = market_return.rolling(window=50).mean()

sma5 = market_return.rolling(window=5).mean()
sma20 = market_return.rolling(window=20).mean()
target = (sma5 > sma20).astype(int).shift(-5).reindex(features.index).fillna(0)

# Train model
X_train = features.loc[features.index.year <= 2024].dropna()
y_train = target.loc[X_train.index]

xgb_model = xgb.XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=5, 
                              random_state=SEED, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

# Optimized Threshold
train_probs = xgb_model.predict_proba(X_train)[:, 1]
opt_t = np.linspace(0.3, 0.7, 41)[np.argmax([fbeta_score(y_train, (train_probs >= t).astype(int), beta=0.5) 
                                              for t in np.linspace(0.3, 0.7, 41)])]

# Generate probabilities
X_all = features.dropna()
all_probs = pd.Series(xgb_model.predict_proba(X_all)[:, 1], index=X_all.index)

print(f"✓ Model trained. Optimized Threshold: {opt_t:.2f}")

✓ Model trained. Optimized Threshold: 0.54


## 3. Portfolio Optimization & Selection Functions

In [4]:
def optimize_markowitz(selected_returns):
    if len(selected_returns.columns) == 0: return {}
    if len(selected_returns.columns) == 1: return {selected_returns.columns[0]: 1.0}
    mu, sigma = selected_returns.mean() * 252, selected_returns.cov() * 252
    num_assets = len(mu)
    def objective(w):
        ret = np.sum(w * mu)
        risk = np.sqrt(np.dot(w.T, np.dot(sigma, w)))
        if risk < 0.0001: return 0
        return -(ret / risk)
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds = tuple((0, 1) for _ in range(num_assets))
    init_guess = [1./num_assets] * num_assets
    res = minimize(objective, init_guess, method='SLSQP', bounds=bounds, constraints=constraints)
    return dict(zip(selected_returns.columns, res.x)) if res.success else {}

def get_assets_graph_diversify(returns_window, corr_threshold=0.4):
    corr_mat = returns_window.corr()
    G = nx.Graph()
    momentum = returns_window.mean()
    assets = list(momentum.sort_values(ascending=False).index)
    G.add_nodes_from(assets)
    for i in range(len(assets)):
        for j in range(i+1, len(assets)):
            if abs(corr_mat.loc[assets[i], assets[j]]) > corr_threshold:
                G.add_edge(assets[i], assets[j])
    return list(nx.approximation.maximum_independent_set(G))

def get_assets_graph_cluster(returns_window, corr_threshold=0.4):
    corr_mat = returns_window.corr()
    G = nx.Graph()
    momentum = returns_window.mean()
    assets = list(momentum.sort_values(ascending=False).index)
    G.add_nodes_from(assets)
    for i in range(len(assets)):
        for j in range(i+1, len(assets)):
            if abs(corr_mat.loc[assets[i], assets[j]]) < corr_threshold:
                G.add_edge(assets[i], assets[j])
    return list(nx.approximation.maximum_independent_set(G))

print("✓ Functions defined")

✓ Functions defined


## 4. Run Simulations (Diversify vs Cluster)

In [5]:
def run_simulation_ai_gated(test_dates, returns, ai_probs, threshold, market_idx, 
                            selection_func, override_ma=None, fee=0.0025, 
                            g_lookback=60, rebal_period=20, turnover_buffer=0.05):
    val = 100.0
    history = [val]
    dates = [test_dates[0]]
    current_weights = {} 
    risk_status = False 
    days_since_rebal = 999
    market_ma = None
    if override_ma:
        market_ma = market_idx.rolling(window=override_ma).mean()
    
    for i, date in enumerate(test_dates[:-1]):
        prev_risk_status = risk_status
        prob = ai_probs.loc[date]
        
        # AI Signal with hysteresis
        ai_signal = False
        buffer = 0.05
        if not risk_status and prob > (threshold + buffer): 
            ai_signal = True
        elif risk_status and prob < (threshold - buffer): 
            ai_signal = False
        else: 
            ai_signal = risk_status
        
        # Trend Override
        final_signal = ai_signal
        if override_ma and market_ma is not None:
            try:
                current_price = market_idx.loc[date]
                ma_price = market_ma.loc[date]
                if not pd.isna(ma_price) and not pd.isna(current_price):
                    is_uptrend = (current_price > ma_price)
                    if ai_signal == False and is_uptrend:
                        final_signal = True
            except KeyError:
                pass
        
        risk_status = final_signal
        regime_changed = (risk_status != prev_risk_status)
        target_weights = current_weights.copy()
        
        if not risk_status:
            if regime_changed or 'CASH' not in current_weights or current_weights.get('CASH', 0) < 0.99:
                target_weights = {'CASH': 1.0}
                days_since_rebal = 0
        else:
            if regime_changed or days_since_rebal >= rebal_period:
                loc_idx = returns.index.get_loc(date)
                window_rets = returns.iloc[max(0, loc_idx-g_lookback):loc_idx]
                selected = selection_func(window_rets)
                optimized_weights = optimize_markowitz(window_rets[selected])
                all_keys = set(list(optimized_weights.keys()) + list(current_weights.keys()))
                turnover_est = sum(abs(optimized_weights.get(k, 0) - current_weights.get(k, 0)) for k in all_keys)
                if regime_changed or turnover_est > turnover_buffer:
                    target_weights = optimized_weights
                    days_since_rebal = 0
            days_since_rebal += 1

        # Execution with transaction costs
        all_keys_exec = set(list(target_weights.keys()) + list(current_weights.keys()))
        turnover_actual = sum(abs(target_weights.get(k, 0) - current_weights.get(k, 0)) for k in all_keys_exec)
        if turnover_actual > 0.001:
            cost = val * turnover_actual * fee
            val -= cost
        
        next_date = test_dates[i+1]
        day_ret = 0
        new_weights_drifted = {}
        
        if 'CASH' in target_weights and target_weights['CASH'] > 0.99:
            day_ret = 0
            new_weights_drifted = {'CASH': 1.0}
        else:
            for asset, w in target_weights.items():
                if asset in returns.columns:
                    r = returns.loc[next_date, asset]
                    r_asset = r if not pd.isna(r) else 0
                    day_ret += w * r_asset
                    new_weights_drifted[asset] = w * (1 + r_asset)
                else:
                    new_weights_drifted[asset] = w
            total_w = sum(new_weights_drifted.values()) if new_weights_drifted else 0
            if total_w > 0:
                new_weights_drifted = {k: v/total_w for k, v in new_weights_drifted.items()}
        
        val *= (1 + day_ret)
        history.append(val)
        dates.append(next_date)
        current_weights = new_weights_drifted
    
    return pd.DataFrame({'Portfolio_Value': history}, index=dates)

print("✓ Simulation function ready")

✓ Simulation function ready


In [6]:
# Run simulations for each year
years = [2023, 2024, 2025]
results = {}

for year in years:
    print(f"Processing {year}...")
    test_dates = returns.loc[returns.index.year == year].index
    
    # Diversify Strategy
    diversify = run_simulation_ai_gated(
        test_dates, returns, all_probs, opt_t, market_index,
        get_assets_graph_diversify, override_ma=200
    )
    
    # Cluster Strategy
    cluster = run_simulation_ai_gated(
        test_dates, returns, all_probs, opt_t, market_index,
        get_assets_graph_cluster, override_ma=200
    )
    
    # Static Markowitz (baseline)
    static = run_simulation_static(
        test_dates, returns, 
        lambda x: list(x.columns)
    )
    
    results[year] = {
        'Diversify': diversify,
        'Cluster': cluster,
        'Static': static
    }

print("\n✓ All simulations completed!")

Processing 2023...


KeyError: Timestamp('2023-04-30 00:00:00')

## 5. Calculate Performance Metrics

In [ ]:
def calculate_metrics(portfolio_df):
    """Calculate performance metrics"""
    values = portfolio_df['Portfolio_Value']
    returns = values.pct_change().dropna()
    
    total_return = (values.iloc[-1] / values.iloc[0] - 1) * 100
    ann_return = ((1 + total_return/100) ** (252/len(returns)) - 1) * 100
    volatility = returns.std() * np.sqrt(252) * 100
    sharpe = ann_return / volatility if volatility > 0 else 0
    
    # Max Drawdown
    cummax = values.cummax()
    drawdown = ((values - cummax) / cummax * 100)
    max_dd = drawdown.min()
    
    return {
        'Total Return (%)': total_return,
        'Ann. Return (%)': ann_return,
        'Volatility (%)': volatility,
        'Sharpe Ratio': sharpe,
        'Max Drawdown (%)': max_dd
    }

# Calculate metrics for all strategies
metrics_summary = []

for year in years:
    for strategy_name in ['Diversify', 'Cluster', 'Static']:
        metrics = calculate_metrics(results[year][strategy_name])
        metrics['Year'] = year
        metrics['Strategy'] = strategy_name
        metrics_summary.append(metrics)

metrics_df = pd.DataFrame(metrics_summary)
metrics_df = metrics_df[['Year', 'Strategy', 'Total Return (%)', 'Ann. Return (%)', 
                         'Volatility (%)', 'Sharpe Ratio', 'Max Drawdown (%)']]

print("\n=== PERFORMANCE METRICS ===")
print(metrics_df.to_string(index=False))

## 6. Visualisasi: Performance Comparison per Year

In [ ]:
# Plot performance for each year
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

for idx, year in enumerate(years):
    ax = axes[idx]
    
    # Plot each strategy
    ax.plot(results[year]['Static'].index, results[year]['Static']['Portfolio_Value'],
            label='Static Markowitz (Baseline)', color='gray', linestyle='-', linewidth=2, alpha=0.7)
    ax.plot(results[year]['Diversify'].index, results[year]['Diversify']['Portfolio_Value'],
            label='AI + Graph Diversify (±0.4)', color='blue', linestyle='-', linewidth=2.5)
    ax.plot(results[year]['Cluster'].index, results[year]['Cluster']['Portfolio_Value'],
            label='AI + Graph Cluster (±0.4)', color='magenta', linestyle='--', linewidth=2.5)
    
    ax.set_title(f'V62 Crypto: AI-Gated Graph Strategies vs Static Markowitz ({year})',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Date', fontsize=10)
    ax.set_ylabel('Portfolio Value', fontsize=10)
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('strategy_comparison_by_year.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Year-by-year comparison saved!")

## 7. Visualisasi: Metrics Comparison (Bar Charts)

In [ ]:
# Compare Diversify vs Cluster only (exclude Static)
comparison_df = metrics_df[metrics_df['Strategy'].isin(['Diversify', 'Cluster'])].copy()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics_to_plot = [
    ('Total Return (%)', 'Total Return Comparison'),
    ('Sharpe Ratio', 'Risk-Adjusted Return Comparison'),
    ('Volatility (%)', 'Volatility Comparison'),
    ('Max Drawdown (%)', 'Max Drawdown Comparison')
]

for idx, (metric, title) in enumerate(metrics_to_plot):
    ax = axes[idx // 2, idx % 2]
    
    pivot = comparison_df.pivot(index='Year', columns='Strategy', values=metric)
    pivot.plot(kind='bar', ax=ax, color=['blue', 'magenta'], width=0.7)
    
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Year', fontsize=10)
    ax.set_ylabel(metric, fontsize=10)
    ax.legend(title='Strategy', fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
    
    # Add value labels on bars
    for container in ax.containers:
        ax.bar_label(container, fmt='%.1f', fontsize=8)

plt.tight_layout()
plt.savefig('diversify_vs_cluster_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Metrics comparison saved!")

## 8. Visualisasi: Average Performance Across Years

In [ ]:
# Calculate average metrics
avg_metrics = comparison_df.groupby('Strategy').mean(numeric_only=True)

print("\n=== AVERAGE PERFORMANCE (2023-2025) ===")
print(avg_metrics.to_string())

In [ ]:
# Horizontal bar chart for average metrics
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(avg_metrics.columns))
width = 0.35

div_values = avg_metrics.loc['Diversify'].values
clus_values = avg_metrics.loc['Cluster'].values

bars1 = ax.barh(x - width/2, div_values, width, label='Diversify', color='blue', edgecolor='black')
bars2 = ax.barh(x + width/2, clus_values, width, label='Cluster', color='magenta', edgecolor='black')

ax.set_yticks(x)
ax.set_yticklabels(avg_metrics.columns, fontsize=10)
ax.set_xlabel('Value', fontsize=11, fontweight='bold')
ax.set_title('Average Performance: Diversify vs Cluster (2023-2025)',
             fontsize=13, fontweight='bold', pad=15)
ax.legend(fontsize=10)
ax.grid(axis='x', alpha=0.3)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        width_val = bar.get_width()
        ax.text(width_val, bar.get_y() + bar.get_height()/2,
                f'{width_val:.2f}',
                ha='left', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('average_performance_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Average performance chart saved!")

## 9. Visualisasi: Win Rate Analysis

In [ ]:
# Calculate which strategy won in each metric for each year
win_count = {'Diversify': {}, 'Cluster': {}}

for metric in ['Total Return (%)', 'Sharpe Ratio']:
    win_count['Diversify'][metric] = 0
    win_count['Cluster'][metric] = 0
    
    for year in years:
        year_data = comparison_df[comparison_df['Year'] == year]
        div_val = year_data[year_data['Strategy'] == 'Diversify'][metric].values[0]
        clus_val = year_data[year_data['Strategy'] == 'Cluster'][metric].values[0]
        
        if div_val > clus_val:
            win_count['Diversify'][metric] += 1
        else:
            win_count['Cluster'][metric] += 1

# For negative metrics (lower is better)
for metric in ['Volatility (%)', 'Max Drawdown (%)']:
    win_count['Diversify'][metric] = 0
    win_count['Cluster'][metric] = 0
    
    for year in years:
        year_data = comparison_df[comparison_df['Year'] == year]
        div_val = year_data[year_data['Strategy'] == 'Diversify'][metric].values[0]
        clus_val = year_data[year_data['Strategy'] == 'Cluster'][metric].values[0]
        
        # For these metrics, lower (less negative) is better
        if div_val < clus_val:
            win_count['Diversify'][metric] += 1
        else:
            win_count['Cluster'][metric] += 1

# Create DataFrame
win_df = pd.DataFrame(win_count).T

print("\n=== WIN RATE (Out of 3 Years) ===")
print(win_df)

In [ ]:
# Visualize win rate
fig, ax = plt.subplots(figsize=(10, 6))

win_df.plot(kind='bar', ax=ax, color=['blue', 'magenta'], width=0.7)
ax.set_title('Strategy Win Rate by Metric (2023-2025)',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Strategy', fontsize=11)
ax.set_ylabel('Number of Years Won', fontsize=11)
ax.set_ylim(0, 3.5)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Metric', fontsize=9, loc='upper right')
ax.grid(axis='y', alpha=0.3)

# Add value labels
for container in ax.containers:
    ax.bar_label(container, fmt='%d', fontsize=9)

plt.tight_layout()
plt.savefig('strategy_win_rate.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Win rate chart saved!")

## 10. Export Results to Excel

In [ ]:
excel_filename = 'strategy_comparison_diversify_vs_cluster.xlsx'

with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
    # Sheet 1: All Metrics
    metrics_df.to_excel(writer, sheet_name='All_Metrics', index=False)
    
    # Sheet 2: Average Metrics
    avg_metrics.to_excel(writer, sheet_name='Average_Metrics')
    
    # Sheet 3: Win Rate
    win_df.to_excel(writer, sheet_name='Win_Rate')
    
    # Sheet 4-6: Portfolio Values per Year
    for year in years:
        combined = pd.DataFrame({
            'Date': results[year]['Diversify'].index,
            'Diversify': results[year]['Diversify']['Portfolio_Value'].values,
            'Cluster': results[year]['Cluster']['Portfolio_Value'].values,
            'Static': results[year]['Static']['Portfolio_Value'].values
        })
        combined.to_excel(writer, sheet_name=f'Portfolio_{year}', index=False)

print(f"\n✓ Results exported to: {excel_filename}")

## 11. Summary & Conclusion

In [ ]:
print("="*70)
print("STRATEGY COMPARISON: DIVERSIFY vs CLUSTER - SUMMARY")
print("="*70)

# Overall winner
total_wins = win_df.sum(axis=1)
winner = total_wins.idxmax()

print(f"\n🏆 OVERALL WINNER: {winner}")
print(f"   → Total wins across all metrics: {total_wins[winner]}/12")

print(f"\n📊 AVERAGE PERFORMANCE (2023-2025):")
print(f"\n   Diversify Strategy:")
print(f"   → Return: {avg_metrics.loc['Diversify', 'Total Return (%)']:.2f}%")
print(f"   → Sharpe: {avg_metrics.loc['Diversify', 'Sharpe Ratio']:.3f}")
print(f"   → Max DD: {avg_metrics.loc['Diversify', 'Max Drawdown (%)']:.2f}%")

print(f"\n   Cluster Strategy:")
print(f"   → Return: {avg_metrics.loc['Cluster', 'Total Return (%)']:.2f}%")
print(f"   → Sharpe: {avg_metrics.loc['Cluster', 'Sharpe Ratio']:.3f}")
print(f"   → Max DD: {avg_metrics.loc['Cluster', 'Max Drawdown (%)']:.2f}%")

print(f"\n🔍 KEY INSIGHTS:")
if avg_metrics.loc['Diversify', 'Sharpe Ratio'] > avg_metrics.loc['Cluster', 'Sharpe Ratio']:
    print("   → Diversify strategy provides better risk-adjusted returns")
else:
    print("   → Cluster strategy provides better risk-adjusted returns")

if abs(avg_metrics.loc['Diversify', 'Max Drawdown (%)']) < abs(avg_metrics.loc['Cluster', 'Max Drawdown (%)']):
    print("   → Diversify strategy has lower maximum drawdown (better capital preservation)")
else:
    print("   → Cluster strategy has lower maximum drawdown (better capital preservation)")

print("\n" + "="*70)
print("✓ Analysis completed successfully!")
print("="*70)

---

## 📁 Output Files Generated:

1. **strategy_comparison_by_year.png** - Performance charts for 2023, 2024, 2025
2. **diversify_vs_cluster_metrics.png** - Metrics comparison bar charts
3. **average_performance_comparison.png** - Average performance horizontal bars
4. **strategy_win_rate.png** - Win rate analysis
5. **strategy_comparison_diversify_vs_cluster.xlsx** - Complete data export

---